In [33]:
#@title The MIT License (MIT)
#
# Copyright (c) 2026 Eric dos Santos.
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in
# all copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
# THE SOFTWARE.

# Fake News Classification

This project aims to develop a neural network for detecting fake news in Portuguese, using the datasets [Fake.br-Corpus](https://github.com/roneysco/Fake.br-Corpus), [FakeTrue.br](https://github.com/jpchav98/FakeTrue.Br) and [FakeRecogna](https://github.com/Gabriel-Lino-Garcia/FakeRecogna). 

With this, we seek to create a system capable of identifying patterns and distinguishing fake news from real news, contributing to the fight against misinformation.

<table class="tfo-notebook-buttons" align="center">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/ericshantos/br_fake_news_detector/blob/main/br_fake_news_detector_model.ipynb
"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run on Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/ericshantos/br_fake_news_detector/blob/main/br_fake_news_detector_model.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View the code on GitHub</a>
  </td>
</table>

## Dataset loading

In [ ]:
from abc import ABC, abstractmethod
from enum import Enum
from pathlib import Path
import pandas as pd
import os

In [ ]:
!git clone https://github.com/roneysco/Fake.br-Corpus
fb_repo = Path("./Fake.br-Corpus/full_texts")

!git clone https://github.com/jpchav98/FakeTrue.Br.git
ft_repo = Path("./FakeTrue.Br")

!git clone https://github.com/Gabriel-Lino-Garcia/FakeRecogna.git
fr_repo = Path("./FakeRecogna/dataset")

fatal: destination path 'Fake.br-Corpus' already exists and is not an empty directory.
fatal: destination path 'FakeTrue.Br' already exists and is not an empty directory.
fatal: destination path 'FakeRecogna' already exists and is not an empty directory.


In [ ]:
class BaseExtractor(ABC):
  def __init__(self, path: Path) -> None:
    self.path = path

  @abstractmethod
  def extract(self, label: int) -> pd.DataFrame:
    raise NotImplementedError

In [ ]:
class Labels(Enum):
  FAKE = 0
  TRUE = 1

  @classmethod
  def to_dict(cls) -> dict[str, int]:
    return {
      "fake": cls.FAKE.value,
      "true": cls.TRUE.value
    }

## Fake.br Extractor

In [ ]:
class FakeBrExtractor(BaseExtractor):
  def __init__(self, path: Path) -> None:
    super().__init__(path)

  def extract(self, labels: int) -> pd.DataFrame:
    news = []

    for dir in os.listdir(self.path):
      if dir not in labels:
        continue

      label = labels[dir]

      for file_path in Path(self.path / dir).glob("*.txt"):
        try:
          content = file_path.read_text(encoding='utf-8')
          news.append({"content": content, "label": label})
        except Exception as e:
          print(f"Error reading file {file_path}: {e}")

    return pd.DataFrame(news)

## FakeTrue.Br Extractor

In [ ]:
class FakeTrueExtractor(BaseExtractor):
  def __init__(self, path: Path) -> None:
    super().__init__(path)

  def extract(self, file: str, label: dict[str, int]) -> pd.DataFrame:
    df_raw = pd.read_csv(self.path / file)

    fake_df = pd.DataFrame()
    fake_df['content'] = df_raw['fake']
    fake_df['label'] = label['fake']

    real_df = pd.DataFrame()
    real_df['content'] = df_raw['true']
    real_df['label'] = label['true']

    final_df = pd.concat([fake_df, real_df], ignore_index=True)

    final_df = final_df.dropna(subset=['content'])

    return final_df

## FakeRecogna Extractor

In [ ]:
class FakeRecognaExtractor(BaseExtractor):
  def __init__(self, path: Path) -> None:
    super().__init__(path)

  def extract(self, file: str, label: dict[str, int]) -> pd.DataFrame:
    df_raw = pd.read_excel(self.path / file)

    news = df_raw[['Noticia', 'Classe']]

    news = news.rename(columns={
        'Noticia': 'content',
        'Classe': 'label'
    })

    news.dropna(subset=['content', 'label'], inplace=True)

    news['content'] = news['content'].astype(str)
    news['label'] = news['label'].astype(int)

    return news

### News content extraction:


In [ ]:
fb_news = FakeBrExtractor(fb_repo).extract(Labels.to_dict())

In [ ]:
ft_news = FakeTrueExtractor(ft_repo).extract(
  "FakeTrueBr_corpus.csv", Labels.to_dict()
)

In [ ]:
fr_news = FakeRecognaExtractor(fr_repo).extract(
  "FakeRecogna.xlsx", Labels.to_dict()
)

## Data preprocessing

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --upgrade

from torch.utils.data import Dataset, DataLoader

Looking in indexes: https://download.pytorch.org/whl/cu121


### Concatenate the DataFrames

Group Dataframes to generate a single robust database.

In [ ]:
data_news = pd.concat(
  [fb_news, ft_news, fr_news], ignore_index=True
).sample(frac=1, random_state=13)

Final base information:

In [ ]:
data_news.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22684 entries, 10328 to 338
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   content  22684 non-null  object
 1   label    22684 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 531.7+ KB


## Training

In [ ]:
from transformers import AutoTokenizer
import torch

### Splitting the dataset into training and testing

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
  data_news['content'],
  data_news['label'],
  test_size=0.1,
  random_state=13,
  stratify=data_news['label']
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Training set size: (20415,)
Test set size: (2269,)


### Create tokenizer

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
  "neuralmind/bert-base-portuguese-cased"
)

## Architectures

In [ ]:
class NewsDataset(Dataset):
  def __init__(self, texts, label, tokenizer, max_length=256):
    self.texts = texts.tolist() if hasattr(texts, 'tolist') else texts
    self.label = label.tolist() if hasattr(label, 'tolist') else label
    self.tokenizer = tokenizer

    self.max_length = max_length

  def __getitem__(self, idx: int):
    encoding = self.tokenizer(
        self.texts[idx],
        padding='max_length',
        truncation=True,
        max_length=self.max_length,
        return_tensors="pt"
    )

    return {
        "input_ids": encoding["input_ids"].squeeze(0),
        "attention_mask": encoding["attention_mask"].squeeze(0),
        "label": torch.tensor(self.label[idx], dtype=torch.long)
    }

  def __len__(self) -> int:
    return len(self.texts)

In [ ]:
dataset_train = NewsDataset(X_train, y_train, tokenizer)
dataset_test = NewsDataset(X_test, y_test, tokenizer)

In [ ]:
loader_train = DataLoader(
  dataset_train,
  batch_size=32,
  shuffle=True,
  pin_memory=True,
  num_workers=4,
  prefetch_factor=2
)

loader_test = DataLoader(
  dataset_test,
  batch_size=32,
  pin_memory=True,
  num_workers=4
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


### Model architecture

In [ ]:
import torch.nn as nn
from transformers import AutoModel

In [ ]:
class BERTClassifier(nn.Module):

  def __init__(self, drop_rate: int = 0.2) -> None:
    super().__init__()

    self.bert = AutoModel.from_pretrained(
      "neuralmind/bert-base-portuguese-cased"
    )

    for param in self.bert.embeddings.parameters():
        param.requires_grad = False

    hidden = self.bert.config.hidden_size

    self.classifier = nn.Sequential(
        nn.Linear(hidden, 32),
        nn.GELU(),
        nn.Dropout(drop_rate),

        nn.Linear(32, 16),
        nn.GELU(),

        nn.Linear(16, 1)
    )

  def forward(self, input_ids, attention_mask):
    outputs = self.bert(
      input_ids=input_ids,
      attention_mask=attention_mask
    )

    pooler = outputs.last_hidden_state[:, 0, :]
    return self.classifier(pooler)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = BERTClassifier().to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Model compilation**:

In [ ]:
import torch.optim as optim

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=5e-5)

### Training the model

In [ ]:
from tqdm import tqdm

In [ ]:
epochs = 5
accumulation_steps = 2

In [ ]:
for epoch in range(epochs):
  model.train()
  total_loss = 0
  optimizer.zero_grad()

  progress_bar = tqdm(loader_train, desc=f"Epoch {epoch+1}")

  for batch_idx, batch in enumerate(progress_bar):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    outputs = model(input_ids, attention_mask).squeeze()
    loss = criterion(outputs, labels.float())

    loss = loss / accumulation_steps
    loss.backward()

    total_loss += loss.item() * accumulation_steps

    if (batch_idx + 1) % accumulation_steps == 0:
      optimizer.step()
      optimizer.zero_grad()

    progress_bar.set_postfix({
      "batch_loss": f"{loss.item() * accumulation_steps:.4f}"
    })

    if batch_idx % 50 == 0:
      torch.cuda.empty_cache()

    if (batch_idx + 1) % accumulation_steps != 0:
      optimizer.step()
      optimizer.zero_grad()

  avg_loss = total_loss / len(loader_train)
  print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

Epoch 1: 100%|██████████| 638/638 [15:16<00:00,  1.44s/it, batch_loss=0.0575]


Epoch 1 | Avg Loss: 0.1505


Epoch 2: 100%|██████████| 638/638 [15:14<00:00,  1.43s/it, batch_loss=0.0146]


Epoch 2 | Avg Loss: 0.0673


Epoch 3: 100%|██████████| 638/638 [15:14<00:00,  1.43s/it, batch_loss=0.0085]


Epoch 3 | Avg Loss: 0.0416


Epoch 4: 100%|██████████| 638/638 [15:15<00:00,  1.43s/it, batch_loss=0.0050]


Epoch 4 | Avg Loss: 0.0256


Epoch 5: 100%|██████████| 638/638 [15:15<00:00,  1.43s/it, batch_loss=0.0055]

Epoch 5 | Avg Loss: 0.0252


## Save to model

In [ ]:
torch.save(model.state_dict(), "veritas-bert-ptbr.pth")

In [ ]:
state_dict = torch.load("veritas-bert-ptbr.pth", map_location=device)

In [ ]:
model = BERTClassifier(drop_rate=0.2)
model.load_state_dict(state_dict)
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af

## Model evaluation

In [ ]:
y_true = []
y_pred = []
y_prob = []

model.eval()

with torch.no_grad():
  for batch in loader_test:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["label"].to(device)

    logits = model(input_ids, attention_mask)

    probs = torch.sigmoid(logits)

    preds = (probs > 0.7).long()

    y_true.extend(labels.cpu().numpy())
    y_pred.extend(preds.cpu().numpy())
    y_prob.extend(probs.cpu().numpy())

In [ ]:
from sklearn.metrics import (
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  confusion_matrix,
  classification_report,
  roc_auc_score
)

In [ ]:
print("Acurácia:", accuracy_score(y_true, y_pred))
print("Precisão:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

print("\nRelatório completo:")
print(classification_report(y_true, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_true, y_pred))

print("\nROC-AUC:")
print(roc_auc_score(y_true, y_prob))

Acurácia: 0.9704715733803437
Precisão: 0.9854413102820746
Recall: 0.955026455026455
F1: 0.9699955217196596

Relatório completo:
              precision    recall  f1-score   support

           0       0.96      0.99      0.97      1135
           1       0.99      0.96      0.97      1134

    accuracy                           0.97      2269
   macro avg       0.97      0.97      0.97      2269
weighted avg       0.97      0.97      0.97      2269


Matriz de confusão:
[[1119   16]
 [  51 1083]]

ROC-AUC:
0.995654538532659
